## Middleware in LangChain
Middleware provides a way to control and customize what happens inside an AI agent during execution. It allows developers to inspect, modify, and manage agent behavior at different stages of its workflow.
#### Key Uses of Middleware
- Logging and Monitoring: Track agent behavior, execution steps, errors, and performance.
- Prompt and Output Transformation: Modify prompts, tool selection, and output formatting.
- Retries and Fallbacks: Retry failed operations, switch to alternative models, or stop execution when necessary.
- Rate Limiting: Control how frequently models or tools can be called.
- Guardrails: Apply rules to restrict unsafe or unwanted behavior.
- PII Detection: Detect or handle personally identifiable information, such as email addresses and phone numbers.
#### How It Works
1. The agent begins execution.
2. Middleware runs at configured stages of the workflow.
3. It inspects or modifies the agent's state, model requests, or tool interactions.
4. The agent continues, retries, or terminates based on the middleware logic.



### Built-in Middleware in LangChain

- **`ModelRetryMiddleware`** — Retries failed LLM calls.
- **`ToolRetryMiddleware`** — Retries failed tool executions.
- **`ToolErrorMiddleware`** — Handles tool errors and returns error messages to the model.
- **`ModelFallbackMiddleware`** — Switches to another model if the primary model fails.
- **`SummarizationMiddleware`** — Summarizes older conversation history to manage context length.
- **`HumanInTheLoopMiddleware`** — Requests human approval before selected tool calls.
- **`ModelCallLimitMiddleware`** — Limits the number of model calls.
- **`ToolCallLimitMiddleware`** — Limits the number of tool calls.
- **`PIIMiddleware`** — Detects and handles sensitive personal information.
- **`TodoListMiddleware`** — Helps an agent plan and track tasks.
- **`LLMToolSelectorMiddleware`** — Selects relevant tools for a user request.
- **`ContextEditingMiddleware`** — Removes or trims tool results to manage context size.

In [46]:
import os
from dotenv import load_dotenv
load_dotenv(r"D:\GenAI\.env", override=True)

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")



## Summarization Middleware

Automatically summarizes conversation history when approaching token limits, preserving recent messages while compressing older context.

##### Use Cases
- Long-running conversations that exceed the context window.
- Multi-turn dialogues with extensive history.
- Applications where preserving conversation context matters.

### Message Lengths

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),  # summarize when length of messages becomes 10
            keep=("messages",4)       # no. of recent messages to keep as it is without summarizing
        )
    ]
)

### Run with thread_id
config = {
    "configurable":{
        "thread_id":"test-1"
    }
}

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='ab57321c-c60c-4f37-b727-1ef5b8f68829'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. Provide answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 78, 'total_tokens': 112, 'completion_time': 0.070500708, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.003010004, 'prompt_tokens_details': None, 'queue_time': 0.394792121, 'total_time': 0.073510712}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a186b3-5f2b-7037-9598-8a9ae5fb73b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 34, 'total_tokens': 112, 'output_token_details': {'reasoning': 15}})]}
Messa

### Tokens

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ~ 1 token

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Find hotels in {city}")]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} - {tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages'][-1]}")

Paris - 183 tokens, 4 messages
content=[{'type': 'text', 'text': 'I found a few hotels in Paris for you:\n\n1.  **Grand Hotel**: 5 stars, $350/night, features a spa, pool, and gym.\n2.  **City Inn**: 4 stars, $180/night, features a business center.\n3.  **Budget Stay**: 3 stars, $75/night, includes free Wi-Fi.\n\nWould you like more options, or perhaps help with booking one of these?', 'extras': {'signature': 'EnMKcQFpFH0Tr6BNbnjFTcxxS1EFMIN6uuTfBSvzW3YhB2VeWOJwcJDvx0m07I22mdT/B72akcMhKlI/3rUZKhUCbx4ZHwqANdfaBiR4hychAVa4Tr9eIX3k8QvE73s0jjRzaEvvhxRpuqF5DZEonk/CCYOs'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a186b3-794d-72a1-be0d-c2b71cad242d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 148, 'output_tokens': 102, 'total_tokens': 250, 'input_token_details': {'cache_read': 0}}
London - 367 tokens, 8 messages
content=[{'type': 'text

### Fractions

In [49]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# Low fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 
    print(f"{city}: ~{tokens} tokens (fraction: {fraction:.4%}, {len(response['messages'])} msgs)")
    print(response['messages'])

Paris: ~109 tokens (fraction: 0.0852%, 4 msgs)
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='6ca86f44-4e13-4fa8-93dc-8692cc9c3c1c'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_327007': 'EnMKcQFpFH0TWV2sc6KdJjyYwzjW5i9VSY4guVbwQ0BxZD3vtbl55ngBpAQ2D0Ioz9R3cyx/WKfmk2qu49gho+8W3HIVt+XSaH6HfdvBSTj3H53+VwnXMprijGFCu2xOhPZLUt+kAn7TYerCMECFVFfrwWNy'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a186b4-0423-73e3-92e3-2114d4b3074e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_327007', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Par

## Human-in-the-Loop Middleware

Pauses agent execution for human approval, editing, or rejection of tool calls before they execute.

### Use Cases
- High-stakes operations requiring human approval (e.g., database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

### Approve Request

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langgraph.types import Command

@tool
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id."""
    return f"Email content for ID: {email_id}"

@tool   
def send_email_tool(recipient:str, subject:str, body:str):
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

# Step 1: Request
config = {"configurable":{"thread_id":"test_approve"}}
result=agent.invoke(
    {
        "messages":[
            HumanMessage(content="Send email to aryan@gmail.com with subject 'Hello' and body 'How are you?'")
        ]
    },
    config=config
)

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    approvedResult = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {approvedResult['messages'][-1].content}")

In [ ]:
result

In [ ]:
resultFinal

### Reject Request

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langgraph.types import Command

@tool
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id."""
    return f"Email content for ID: {email_id}"

@tool   
def send_email_tool(recipient:str, subject:str, body:str):
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

# Step 1: Request
config = {"configurable":{"thread_id":"test_reject"}}
result=agent.invoke(
    {
        "messages":[
            HumanMessage(content="Send email to aryan@gmail.com with subject 'Hello' and body 'How are you?'")
        ]
    },
    config=config
)

# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Rejecting...")

    rejectedResult = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {rejectedResult['messages'][-1].content}")

⏸️ Paused! Rejecting...
✅ Result: I’m unable to send the email because the request to use the email‑sending tool was rejected. If you’d still like me to send the message, just let me know and I’ll try again.


In [ ]:
result

In [ ]:
rejectedResult

### Edit Request

In [66]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langgraph.types import Command

@tool
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id."""
    return f"Email content for ID: {email_id}"

@tool   
def send_email_tool(recipient:str, subject:str, body:str):
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

# Step 1: Request
config = {"configurable":{"thread_id":"test_edit"}}
result=agent.invoke(
    {
        "messages":[
            HumanMessage(content=("Send email to aryan@gmail.com with subject 'Hello' and body 'How are you?'"))
        ]
    },
    config=config
)

# Step 2: Edit and Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    editedResult = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"jaiswal@gmail.com",
                                "subject":"Corrected Subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {editedResult['messages'][-1].content}")

⏸️ Paused! Editing...
✅ Result: I attempted to send the email you requested, but a human reviewer intervened and changed the recipient, subject, and body before the message was actually sent. The email that was sent went to **jaiswal@gmail.com** with the subject **“Corrected Subject”** and the body **“This was edited by human before sending.”**

If you’d still like the original message sent to **aryan@gmail.com** (subject “Hello”, body “How are you?”), just let me know and I’ll take care of it.


In [67]:
result

{'messages': [HumanMessage(content="Send email to aryan@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0cc3ad0a-dd47-4110-b083-827d82e51234'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. Use send_email_tool with recipient, subject, body.', 'tool_calls': [{'id': 'fc_5d9e2a25-d83e-4346-9928-caadd120c9ba', 'function': {'arguments': '{"body":"How are you?","recipient":"aryan@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 175, 'total_tokens': 242, 'completion_time': 0.139762984, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.008533133, 'prompt_tokens_details': None, 'queue_time': 0.326133222, 'total_time': 0.148296117}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_47082602e2', 'service_tier': 'on_demand', 'finish_reason': 'tool

In [68]:
editedResult

{'messages': [HumanMessage(content="Send email to aryan@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0cc3ad0a-dd47-4110-b083-827d82e51234'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. Use send_email_tool with recipient, subject, body.', 'tool_calls': [{'id': 'fc_5d9e2a25-d83e-4346-9928-caadd120c9ba', 'function': {'arguments': '{"body":"How are you?","recipient":"aryan@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 175, 'total_tokens': 242, 'completion_time': 0.139762984, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.008533133, 'prompt_tokens_details': None, 'queue_time': 0.326133222, 'total_time': 0.148296117}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_47082602e2', 'service_tier': 'on_demand', 'finish_reason': 'tool